# Datenvorbereitung für den ersten Modellversuch

Dieses Notebook bereitet die untersuchten NPPAD-Dateien für
einen LSTM-Autoencoder vor.

Die Normalbetriebs-Datei wird chronologisch in Training,
Validierung und Test aufgeteilt. Zwischen den Abschnitten
werden Datenpunkte ausgelassen.

Die Normalisierung wird ausschließlich anhand des Trainingsabschnitts
berechnet. Anschließend werden Zeitfenster erzeugt.

## Einschränkung dieses Vorversuchs

Die drei Normalbetriebs-Abschnitte stammen aus derselben Datei.
Sie sind keine unabhängigen Simulationsläufe.

Dieser Vorversuch prüft zunächst die technische Vorgehensweise.
Die in L3 geplante Validierung an getrennten Simulationen
ist damit noch nicht umgesetzt.

Für die LOCA- und SGATR-Dateien werden zunächst Fenster ohne
Ereignislabels vorbereitet. Die Unfallzeitpunkte müssen vor
einer Bewertung der Erkennungsleistung überprüft werden.

In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd

output = Path("data/prepared")
output.mkdir(parents=True, exist_ok=True)

print("Arbeitsordner:", Path.cwd())

In [ ]:
sources = pd.read_csv("data_sources.csv").set_index("scenario")

datasets = {}

for name in ["Normal", "LOCA", "SGATR"]:
    file = Path("data") / f"{name}.csv"

    actual_hash = hashlib.sha256(file.read_bytes()).hexdigest()

    if actual_hash != sources.loc[name, "sha256"]:
        raise ValueError(f"Originaldatei wurde veraendert: {file}")

    frame = pd.read_csv(file).apply(
        pd.to_numeric,
        errors="raise"
    )

    if not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f"Fehlende oder ungueltige Werte in {name}")

    steps = frame["TIME"].diff().dropna()

    if len(steps) == 0 or not (steps > 0).all() or steps.nunique() != 1:
        raise ValueError(f"Zeitstempel pruefen: {name}")

    datasets[name] = frame

normal = datasets["Normal"]

for name, frame in datasets.items():
    if list(frame.columns) != list(normal.columns):
        raise ValueError(f"Unterschiedliche Spalten bei {name}")

print("Originaldateien geprueft und geladen.")

In [ ]:
WINDOW_SIZE = 12
GAP = 12

n = len(normal)

train_end = int(n * 0.6)
val_end = int(n * 0.8)

train = normal.iloc[:train_end].copy()

val = normal.iloc[
    train_end + GAP:val_end
].copy()

test_normal = normal.iloc[
    val_end + GAP:
].copy()

if min(len(train), len(val), len(test_normal)) < WINDOW_SIZE:
    raise ValueError("Zu wenige Daten fuer die gewaehlte Aufteilung.")

split = normal[["TIME"]].copy()

split.insert(0, "source_row", normal.index)
split["partition"] = "gap"

split.loc[train.index, "partition"] = "train"
split.loc[val.index, "partition"] = "validation"
split.loc[test_normal.index, "partition"] = "test_normal"

split.to_csv(output / "normal_split.csv", index=False)

print("Training:", len(train))
print("Validierung:", len(val))
print("Normaler Test:", len(test_normal))
print("Ausgelassene Zeilen:", int((split["partition"] == "gap").sum()))

In [ ]:
features = normal.columns.drop("TIME").tolist()

mean = train[features].mean()
std = train[features].std(ddof=0)

constant = train[features].nunique() == 1

scale = std.mask(
    constant | (std == 0),
    1.0
)


def normalize(frame):
    values = (
        (frame[features] - mean) / scale
    ).to_numpy(dtype=np.float32)

    if not np.isfinite(values).all():
        raise ValueError("Ungueltige Werte nach der Normalisierung.")

    return values


normalization = pd.DataFrame({
    "feature": features,
    "mean": mean.to_numpy(),
    "scale": scale.to_numpy(),
    "constant_in_training": constant.to_numpy()
})

normalization.to_csv(
    output / "normalization.csv",
    index=False
)

print("Eingangsgroessen:", len(features))
print("Im Training konstant:", int(constant.sum()))
print("Normalisiertes Training:", normalize(train).shape)

In [ ]:
def make_windows(frame):
    values = normalize(frame)

    windows = []

    for start in range(len(values) - WINDOW_SIZE + 1):
        window = values[start:start + WINDOW_SIZE]
        windows.append(window)

    if not windows:
        raise ValueError(
            "Dieser Abschnitt ist fuer ein Zeitfenster zu kurz."
        )

    return np.array(windows, dtype=np.float32)

In [ ]:
partitions = {
    "normal_train": train,
    "normal_val": val,
    "normal_test": test_normal,
    "LOCA": datasets["LOCA"],
    "SGATR": datasets["SGATR"]
}

arrays = {}
summary_rows = []
metadata_rows = []

for name, frame in partitions.items():
    arrays[name] = make_windows(frame)

    shape = arrays[name].shape

    summary_rows.append({
        "partition": name,
        "windows": shape[0],
        "time_steps": shape[1],
        "features": shape[2]
    })

    source = "Normal" if name.startswith("normal_") else name

    for start in range(shape[0]):
        end = start + WINDOW_SIZE - 1

        metadata_rows.append({
            "partition": name,
            "window_id": start,
            "source": source,
            "start_row": int(frame.index[start]),
            "end_row": int(frame.index[end]),
            "start_time": float(frame["TIME"].iloc[start]),
            "end_time": float(frame["TIME"].iloc[end])
        })

window_summary = pd.DataFrame(summary_rows)
window_metadata = pd.DataFrame(metadata_rows)

window_summary

In [ ]:
assert set(train.index).isdisjoint(val.index)
assert set(train.index).isdisjoint(test_normal.index)
assert set(val.index).isdisjoint(test_normal.index)

for values in arrays.values():
    assert values.ndim == 3
    assert values.shape[1] == WINDOW_SIZE
    assert values.shape[2] == len(features)
    assert np.isfinite(values).all()

np.savez_compressed(
    output / "windows.npz",
    **arrays,
    feature_names=np.array(features, dtype=str),
    window_size=np.array(WINDOW_SIZE),
    gap=np.array(GAP)
)

window_summary.to_csv(
    output / "window_summary.csv",
    index=False
)

window_metadata.to_csv(
    output / "window_metadata.csv",
    index=False
)

print("Aufteilung und Fenster geprueft und gespeichert.")

## Ergebnis der Datenvorbereitung

Die Normalbetriebs-Datei wurde chronologisch aufgeteilt.
Zwischen den Abschnitten wurden jeweils zwölf Datenpunkte ausgelassen.

Die Normalisierung verwendet ausschließlich Mittelwerte und
Standardabweichungen des Trainingsabschnitts. Alle 96 Prozessgrößen
bleiben zunächst erhalten. Für im Training konstante Größen
wurde der Divisor auf 1 gesetzt.

Die Zeitfenster enthalten jeweils zwölf Messpunkte.
Sie werden innerhalb der bereits getrennten Abschnitte erzeugt.
Kein Fenster überschreitet eine Abschnittsgrenze.

Es entstanden:
- 170 normale Trainingsfenster,
- 37 normale Validierungsfenster,
- 38 normale Testfenster,
- 464 LOCA-Fenster,
- 540 SGATR-Fenster.

Die Unfallfenster besitzen noch keine überprüften Ereignislabels.

Dieser Aufbau ist ein kleiner Vorversuch innerhalb eines
Normalbetriebs-Laufs. Er belegt keine Generalisierung auf
unabhängige Simulationen. Überlappende Fenster sind außerdem
keine unabhängigen statistischen Beobachtungen.